# Sentence-End LSTM — Sequence Model

Локальные фичи (пауза, энергия, питч) дают AUC ~0.664 — потолок точечной модели.
LSTM видит **последовательность** межсловесных границ:
- «темп замедляется 5 слов подряд → конец»
- «питч падает уже 1.5с → конец»
- «пауза вдруг большая → конец» (на фоне истории, где пауз не было)

**Архитектура:** unidirectional LSTM (causal — нет future context, работает real-time)
```
input: [batch, seq_len, 22]  — teacher features (acoustic + MFA)
LSTM:  input=22, hidden=64, layers=2, dropout=0.3
out:   sigmoid(Linear(64 → 1)) per timestep
```
**CoreML export:** stateful (h, c как вход/выход) → Swift держит hidden state между словами.

In [ ]:
!pip install -q torch coremltools scikit-learn pandas numpy matplotlib

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Загрузка и подготовка последовательностей

In [ ]:
import os, urllib.request

REPO = 'https://raw.githubusercontent.com/russianoracle/sentence-end-training/main'
CSV  = 'sentence_end_teacher_v3.csv'

if not os.path.exists(CSV):
    print(f'Downloading {CSV}...')
    urllib.request.urlretrieve(f'{REPO}/{CSV}', CSV)

df = pd.read_csv(CSV)

ACOUSTIC = ['pause_sec','log_pause','rms_before','rms_after','rms_ratio',
            'energy_slope','rate_before','rate_after','rate_delta',
            'vowel_energy_ratio','spectral_flux','pitch_mean','pitch_drop','voiced_fraction']
MFA_REAL  = ['phone_dur_ratio','last_phone_dur','last_vowel_dur','vowel_ratio',
             'n_phones','phone_variability','word_dur','word_dur_ratio_mfa']
FEATURES  = ACOUSTIC + MFA_REAL  # 22 teacher features

print(f'Rows: {len(df):,}  features: {len(FEATURES)}  pos: {df["y"].mean()*100:.1f}%')
print(f'Sources: {df["source"].unique().tolist()}')

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Clip phone_dur_ratio outliers (MFA artifact: pause absorbed into last phone)
df['phone_dur_ratio'] = df['phone_dur_ratio'].clip(0, 5)
df[FEATURES] = df[FEATURES].fillna(0)

# Fit scaler on all data, apply per-source
scaler = StandardScaler()
df[FEATURES] = scaler.fit_transform(df[FEATURES])

# Build sequences per source — sliding window SEQ_LEN, stride STRIDE
SEQ_LEN = 30
STRIDE   = 5

seqs_X, seqs_y, seqs_src = [], [], []
for src, grp in df.groupby('source'):
    X = grp[FEATURES].values.astype(np.float32)
    y = grp['y'].values.astype(np.float32)
    for start in range(0, len(X) - SEQ_LEN + 1, STRIDE):
        seqs_X.append(X[start:start+SEQ_LEN])
        seqs_y.append(y[start:start+SEQ_LEN])
        seqs_src.append(src)

seqs_X = np.array(seqs_X)  # [N, SEQ_LEN, 22]
seqs_y = np.array(seqs_y)  # [N, SEQ_LEN]
print(f'Sequences: {len(seqs_X):,}  shape: {seqs_X.shape}')
print(f'Positive rate: {seqs_y.mean()*100:.1f}%')

In [ ]:
# Train/val split — по source (не перемешиваем временные ряды)
# dudh (~50% данных) → train; остальные → val
train_idx = [i for i, s in enumerate(seqs_src) if s == 'dudh']
val_idx   = [i for i, s in enumerate(seqs_src) if s != 'dudh']

X_tr = torch.tensor(seqs_X[train_idx])
y_tr = torch.tensor(seqs_y[train_idx])
X_va = torch.tensor(seqs_X[val_idx])
y_va = torch.tensor(seqs_y[val_idx])

print(f'Train: {len(X_tr):,} seqs (dudh)')
print(f'Val:   {len(X_va):,} seqs (out-of-domain)')

class SeqDataset(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_dl = DataLoader(SeqDataset(X_tr, y_tr), batch_size=256, shuffle=True)
val_dl   = DataLoader(SeqDataset(X_va, y_va), batch_size=256, shuffle=False)

## 2. LSTM модель

In [ ]:
class SentenceEndLSTM(nn.Module):
    def __init__(self, input_size=22, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x, state=None):
        out, state = self.lstm(x, state)
        logits = self.head(out).squeeze(-1)  # [batch, seq_len]
        return logits, state

model = SentenceEndLSTM(input_size=len(FEATURES)).to(DEVICE)
print(model)
params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {params:,}')

## 3. Обучение

In [ ]:
pos_rate = seqs_y.mean()
pos_weight = torch.tensor([(1 - pos_rate) / pos_rate], device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

def eval_auc(dl):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for xb, yb in dl:
            xb = xb.to(DEVICE)
            logits, _ = model(xb)
            probs = torch.sigmoid(logits).cpu().numpy().ravel()
            all_probs.append(probs)
            all_labels.append(yb.numpy().ravel())
    return roc_auc_score(np.concatenate(all_labels), np.concatenate(all_probs))

train_aucs, val_aucs = [], []
EPOCHS = 50

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits, _ = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    if (epoch + 1) % 5 == 0:
        tr_auc = eval_auc(train_dl)
        va_auc = eval_auc(val_dl)
        train_aucs.append(tr_auc)
        val_aucs.append(va_auc)
        print(f'Epoch {epoch+1:3d}  loss={total_loss/len(train_dl):.4f}  '
              f'train_AUC={tr_auc:.4f}  val_AUC={va_auc:.4f}')

print(f'\nFinal val AUC: {val_aucs[-1]:.4f}')

In [ ]:
plt.figure(figsize=(8, 4))
epochs_plot = list(range(5, EPOCHS+1, 5))
plt.plot(epochs_plot, train_aucs, label='train')
plt.plot(epochs_plot, val_aucs,   label='val')
plt.xlabel('Epoch'); plt.ylabel('AUC-ROC')
plt.title('LSTM training curve')
plt.legend(); plt.tight_layout(); plt.show()

## 4. CoreML export (stateful)

Stateful LSTM: h и c как входы/выходы → Swift держит state между словами.
На каждой обнаруженной границе слова: `predict(featureVector)` → P(sentence_end).
Сброс state при долгой паузе (>2s) — новое высказывание.

In [ ]:
import coremltools as ct
import json

model.eval()

# Wrapper: single timestep inference with explicit state IO
class LSTMStep(nn.Module):
    def __init__(self, lstm_model):
        super().__init__()
        self.lstm = lstm_model.lstm
        self.head = lstm_model.head

    def forward(self, x, h, c):
        # x: [1, 1, features]  h/c: [num_layers, 1, hidden]
        out, (h_new, c_new) = self.lstm(x, (h, c))
        prob = torch.sigmoid(self.head(out)).squeeze()  # scalar
        return prob, h_new, c_new

step_model = LSTMStep(model)
step_model.eval()

NL, H, F = model.num_layers, model.hidden_size, len(FEATURES)
x_dummy = torch.zeros(1, 1, F)
h_dummy = torch.zeros(NL, 1, H)
c_dummy = torch.zeros(NL, 1, H)

traced = torch.jit.trace(step_model, (x_dummy, h_dummy, c_dummy))

ml_model = ct.convert(
    traced,
    inputs=[
        ct.TensorType(name='features', shape=(1, 1, F)),
        ct.TensorType(name='h_in',     shape=(NL, 1, H)),
        ct.TensorType(name='c_in',     shape=(NL, 1, H)),
    ],
    outputs=[
        ct.TensorType(name='prob'),
        ct.TensorType(name='h_out'),
        ct.TensorType(name='c_out'),
    ],
    minimum_deployment_target=ct.target.macOS13,
)

# Embed scaler params as metadata
ml_model.user_defined_metadata['scaler_mean']  = json.dumps(scaler.mean_.tolist())
ml_model.user_defined_metadata['scaler_scale'] = json.dumps(scaler.scale_.tolist())
ml_model.user_defined_metadata['features']     = json.dumps(FEATURES)
ml_model.user_defined_metadata['val_auc']      = str(round(val_aucs[-1], 4))

ml_model.save('SentenceEndLSTM.mlpackage')
print('Saved: SentenceEndLSTM.mlpackage')
print(f'Val AUC: {val_aucs[-1]:.4f}')

try:
    from google.colab import files
    import shutil
    shutil.make_archive('SentenceEndLSTM', 'zip', '.', 'SentenceEndLSTM.mlpackage')
    files.download('SentenceEndLSTM.zip')
except:
    pass

## 5. Верификация — stateful inference

Проверяем что hidden state сохраняется между вызовами и модель работает пошагово.

In [ ]:
import coremltools as ct
import numpy as np
import json

loaded = ct.models.MLModel('SentenceEndLSTM.mlpackage')

NL = model.num_layers
H  = model.hidden_size
F  = len(FEATURES)

h = np.zeros((NL, 1, H), dtype=np.float32)
c = np.zeros((NL, 1, H), dtype=np.float32)

# Probs for first 20 boundaries of val sequence
sample_seq = X_va[0].numpy()  # [SEQ_LEN, F]
sample_lbl = y_va[0].numpy()

print('Step-by-step inference (stateful):')
print(f'{"t":>4}  {"prob":>6}  {"label":>6}  {"h_changed":>12}')
h_prev = h.copy()
for t in range(min(20, len(sample_seq))):
    x_in = sample_seq[t:t+1][np.newaxis]  # [1, 1, F]
    out = loaded.predict({'features': x_in, 'h_in': h, 'c_in': c})
    prob  = float(out['prob'])
    h     = out['h_out']
    c     = out['c_out']
    h_diff = np.abs(h - h_prev).max()
    h_prev = h.copy()
    print(f'{t:>4}  {prob:>6.4f}  {int(sample_lbl[t]):>6}  {h_diff:>12.6f}')

print('\nStateful inference OK — hidden state updates at each step')